### Иттераторы и генераторы

In [12]:
num = [10, 20, 30]
for n in num:
    print(n)

for n in num:
    print(n)
for n in num:
    print(n)

10
20
30
10
20
30
10
20
30


In [2]:
# iter(obj) -> __iter__
# next(it) -> __next__ -> StopIteration

In [6]:
nums = [10, 20, 30]
it = iter(nums)
print(type(nums), type(it))
print(next(it))
print(next(it))
print(next(it))
try:
    print(next(it))
except StopIteration:
    print('Иттерация закончена/элементов больше нет')

<class 'list'> <class 'list_iterator'>
10
20
30
Иттерация закончена/элементов больше нет


In [7]:
nums = [10, 20, 30]
it = iter(nums)
while True:
    try:
        n = next(it)
    except StopIteration:
        break
    print(n)

10
20
30


iter() от итератора возвращает **его же**

iter() от списка каждый раз создает **новый** итератор. По списку можно пройтись сколько угодно раз, а по итератору только один.

In [10]:
nums = [10, 20, 30]
it = iter(nums)
print(iter(it) is it)
print(iter(nums) is iter(nums))
print(hasattr(nums, '__next__'), hasattr(it, '__next__'))

True
False
False True


In [13]:
# map, filter, zip, enumirate, reversed, open file

In [21]:
squares = map(lambda x: x * x, [1, 2, 3])
print(list(squares))
print(list(squares))  # итератор исчерпан

[1, 4, 9]
[]


In [22]:
class Countdow:
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value

In [23]:
c = Countdow(3)
print(list(c))
print(list(c))

[3, 2, 1]
[]


In [24]:
c = Countdow(2)
for a in c:
    for b in c:
        print(a, b)


2 1


In [26]:
for i in range(1, 10):  # list
    print(i)

1
2
3
4
5
6
7
8
9


In [28]:
class CountdowRange:
    def __init__(self, start):
        self.start = start

    def __iter__(self):
        return Countdow(self.start)

r = CountdowRange(2)
print(list(r), list(r))

for a in r:
    for b in r:
        print(a, b)

[2, 1] [2, 1]
2 2
2 1
1 2
1 1


In [30]:
import random
random.seed(1)
rolls = iter(lambda: random.randint(1, 6), 6)
print(list(rolls))
# iter(lambda: f.read(4096), b'')

[2, 5, 1, 3, 1, 4, 4, 4]


#### Генераторы

In [31]:
# yield

In [35]:
def countdown(start: int):
    print('старт генератора')
    while start > 0:
        print(f'перед yield {start}')
        yield start
        print(f'после yield {start}')
        start -= 1
    print('конец генератора')

gen = countdown(2)
print(gen)
print()
print('next ->', next(gen))
print('next ->', next(gen))
print()
try:
    print('next ->', next(gen))
except StopIteration:
    print('значений больше нет')

<generator object countdown at 0x10e56a740>

старт генератора
перед yield 2
next -> 2
после yield 2
перед yield 1
next -> 1

после yield 1
конец генератора
значений больше нет


In [36]:
from inspect import getgeneratorstate

def countdown(start: int):
    while start > 0:
        yield start
        start -= 1

gen = countdown(2)
print(getgeneratorstate(gen))
next(gen)
print(getgeneratorstate(gen), gen.gi_frame.f_locals)
list(gen)
print(getgeneratorstate(gen))

GEN_CREATED
GEN_SUSPENDED {'start': 2}
GEN_CLOSED


In [38]:
class Countdown: # здесь не нужен
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value


class CountdownRange:
    def __init__(self, start):
        self.start = start

    def __iter__(self):
        current = self.start
        while current > 0:
            yield current
            current -= 1

r = CountdownRange(3)
print(list(r), list(r))

[3, 2, 1] [3, 2, 1]


In [39]:
def gen():
    yield 1
    return 'готово'

g = gen()
next(g)
try:
    next(g)
except StopIteration as e:
    print('value:', e.value)

value: готово


In [42]:
import sys
lst = [x * x for x in range(1_000_000)]
gen = (x * x for x in range(1_000_000))
print(sum(x * x for x in range(1_000_000)))
print(f'list: {sys.getsizeof(lst):>10,} байт')
print(f'gen: {sys.getsizeof(gen):>10,} байт')

333332833333500000
list:  8,448,728 байт
gen:        208 байт


In [45]:
import tracemalloc, time
for name, make in [
    ('list', lambda: sum([x * x for x in range(5_000_000)])),
    ('gen', lambda: sum(x * x for x in range(5_000_000))),
]:
    tracemalloc.start()
    t = time.perf_counter()
    make()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    print(f'{name}: пик {peak / 1024 / 1024:6.1f} МБ {time.perf_counter() - t:.2f} сек')

list: пик  195.3 МБ 1.47 сек
gen: пик    0.0 МБ 1.13 сек


In [46]:
# yield from другой_итерируемый_объект - выдает все его элементы по очереди (для рекурсии)

### Ленивые вычисления и пайплайны

In [50]:
def naturals():
    n = 1
    while True:
        yield n
        n += 1

from itertools import islice

print(list(islice(naturals(), 5)))

def fib():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

print(list(islice(fib(), 10)))
print(next(x for x in fib() if x > 10000))

[1, 2, 3, 4, 5]
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
10946


In [52]:
def source():
    for x in [1, 2, 3]:
        print(f'source -> {source}')
        yield x

def square(it):
    for x in it:
        print(f'square {x} -> {x * x}')
        yield x * x

def only_odd(it):
    for x in it:
        if x % 2:
            print(f'only_odd пропустил {x}')
            yield x

pipeline = only_odd(square(source()))
print('пайплайн собран, ничего не выполнено')

пайплайн собран, ничего не выполнено


In [53]:
for r in pipeline:
    print('результат:', r)

source -> <function source at 0x110d4db10>
square 1 -> 1
only_odd пропустил 1
результат: 1
source -> <function source at 0x110d4db10>
square 2 -> 4
source -> <function source at 0x110d4db10>
square 3 -> 9
only_odd пропустил 9
результат: 9


In [59]:
import random, os
random.seed(42)
levels = ['INFO'] * 90 + ['WARNING'] * 8 + ['ERROR'] * 2
services = ['auth', 'api', 'db', 'cache']
with open('app.log', 'w') as f:
    for i in range(5_000_000):
        f.write(f'2026-10-10 13:{i //60 % 60:02d}:{i % 60:02d} {random.choice(levels)} {random.choice(services)} request_id={i}\n')
print(f'{os.path.getsize('app.log') / 1024 / 1024:.0f} МБ')

231 МБ


In [56]:
import tracemalloc
from collections import Counter

def read_lines(path):
    with open(path) as f:
        for line in f:
            yield line.rstrip('\n')

def parse(lines):
    for line in lines:
        date, time, level, service, rest = line.split(' ', 4)
        yield {'level': level, 'service': service}

def only_level(records, level):
    return (r for r in records if r['level'] == level)


In [60]:
tracemalloc.start()
errors = only_level(parse(read_lines('app.log')), 'ERROR')
print(Counter(r['service'] for r in errors))
_, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f'пик памяти: {peak / 1024 / 1024:.2f} МБ')

Counter({'auth': 25001, 'db': 24978, 'cache': 24857, 'api': 24844})
пик памяти: 0.15 МБ


In [ ]:
# map, zip, filter, open etc.

In [4]:
# chain - склеивает несколько источников

from itertools import chain

logs_monday = ['ok', 'error']
logs_tuesday = ('ok', 'ok')

print(list(chain(logs_monday, logs_tuesday, ['timeout'])))

weeks = [[1, 2], [3], [4, 5, 6]]
print(list(chain.from_iterable(weeks)))
# chain.from_iterable(read_lines(p) for p in paths)

['ok', 'error', 'ok', 'ok', 'timeout']
[1, 2, 3, 4, 5, 6]


In [8]:
# islice - срез для итератора
l = [1, 2, 3]
l[1:2]  # gen[10:20] - error
from itertools import islice, count
print(list(islice(range(100), 5)))
print(list(islice(count(1), 3)))

[0, 1, 2, 3, 4]
[1, 2, 3]


In [12]:
# takewhile и dropwhile
from itertools import takewhile, dropwhile
lines = ['# config', '# version 2', 'host=localhost', 'port=8080', '# end']
print(list(dropwhile(lambda l: l.startswith('#'), lines)))
print(list(takewhile(lambda l: l.startswith('#'), lines)))
print([l for l in lines if not l.startswith('#')])

['host=localhost', 'port=8080', '# end']
['# config', '# version 2']
['host=localhost', 'port=8080']


In [22]:
from itertools import pairwise, accumulate
prices = [100, 104, 101, 110, 108]
print([b - a for a, b in pairwise(prices)])

ops = [1000, -200, -300, 500, -1200]
print(list(accumulate(ops)))
print(list(accumulate(prices, max)))

print(next(i for i, b in enumerate(accumulate(ops)) if b <0))

[4, -3, 9, -2]
[1000, 800, 500, 1000, -200]
[100, 104, 104, 110, 110]
4


In [23]:
from itertools import batched, groupby

ids = range(1, 11)
for batch in batched(ids, 4):
    print('отправили в БД:', batch)

отправили в БД: (1, 2, 3, 4)
отправили в БД: (5, 6, 7, 8)
отправили в БД: (9, 10)


In [24]:
evets = [
    ('10:01', 'ERROR'),
    ('10:01', 'ERROR'),
    ('10:04', 'ERROR'),
    ('10:04', 'INFO'),
    ('10:10', 'ERROR'),
]
for minute, group in groupby(evets, key=lambda e: e[0]):
    print(minute, [lvl for _, lvl in group])

10:01 ['ERROR', 'ERROR']
10:04 ['ERROR', 'INFO']
10:10 ['ERROR']


In [26]:
from itertools import tee
gen = (x * x for x in range(5))
a, b = tee(gen)
print(list(a), sum(b))

[0, 1, 4, 9, 16] 30


In [29]:
def checks():
    for x in [1, 2, -3, 4, 5]:
        print('првоеряем', x)
        yield x

print(all(x > 0 for x in checks()))
print(any(x < 0 for x in checks()))

првоеряем 1
првоеряем 2
првоеряем -3
False
првоеряем 1
првоеряем 2
првоеряем -3
True


In [30]:
import random
random.seed(7)
with open('app2.log', 'w') as f:
    for minute in range(60):
        n_err = 30 if minute in (17, 18, 42) else random.randint(0, 4)
        for _ in range(n_err):
            f.write(f'2026-10-17 11:{minute:02d}:{random.randint(0, 59):02d} ERROR api timeout\n')
        for _ in range(50):
            f.write(f'2026-10-17 11:{minute:02d}:{random.randint(0, 59):02d} INFO api ok\n')

In [31]:
from itertools import groupby, pairwise

def read_lines(path):
    with open(path) as f:
        for line in f:
            yield line.rstrip('\n')

def parse(lines):
    for line in lines:
        date, time, level, service, message = line.split(' ', 4)
        yield {'minute': time[:5], 'level': level}

def errors_per_minute(records):
    for minute, group in groupby(records, key=lambda r: r['minute']):
        yield minute, sum(1 for r in group if r['level'] == 'ERROR')

def spikes(counts, factor=3, min_errors=10):
    for (m1, c1), (m2, c2) in pairwise(counts):
        if c2 >= min_errors and c2 >= c1 * factor:
            yield m2, c1, c2

pipline = spikes(errors_per_minute(parse(read_lines('app2.log'))))

In [33]:
for minute, before, now in pipline:
    print(f'всплеск в {minute}: {before} -> {now} ошибок')

всплеск в 11:17: 4 -> 30 ошибок
всплеск в 11:42: 0 -> 30 ошибок


In [34]:
# Арифметика
class Vector:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __repr__(self):
        return f'Vector({self.x}, {self.y})'

    def __eq__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

    def __hash__(self):
        return hash((self.x, self.y))

    def __add__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        return Vector(self.x + other.x, self.y + other.y)

    def __mul__(self, k):
        if not isinstance(k, (int, float)):
            return NotImplemented
        return Vector(self.x * k, self.y * k)

    def __rmul__(self, k):
        return self * k

    def __abs__(self):
        return (self.x ** 2 + self.y ** 2) ** 0.5

    def __bool__(self):
        return bool(self.x or self.y)


a, b = Vector(1, 2), Vector(3, 4)
print(a + b, a * 3, 3 * a, abs(b))
print(a == Vector(1, 2), a == (1, 2), bool(Vector(0, 0)))


try:
    sum([a, b])
except TypeError as e:
    print('TypeError:', e)
print(sum([a, b], Vector(0, 0)))

In [35]:
# сортировка
from functools import total_ordering

@total_ordering
class Version:
    def __init__(self, s):
        self.parts = tuple(int(p) for p in s.split('.'))

    def __repr__(self):
        return f"Version('{'.'.join(map(str, self.parts))}')"

    def __eq__(self, other):
        return self.parts == other.parts

    def __lt__(self, other):
        return self.parts < other.parts


vs = [Version('1.10.0'), Version('1.2.0'), Version('1.9.3')]
print(sorted(vs))
print(max(vs), Version('1.2') <= Version('1.10'))
print(sorted(['1.10.0', '1.2.0', '1.9.3']))

In [36]:
# протокол контейнера
class Playlist:
    def __init__(self, name, songs):
        self.name = name
        self._songs = list(songs)

    def __len__(self):
        return len(self._songs)

    def __getitem__(self, index):
        return self._songs[index]


p = Playlist('road', ['Song A', 'Song B', 'Song C'])
print(len(p), p[0], p[-1], p[1:])
print('Song B' in p)
for song in p:
    print(song)
print(list(reversed(p)))
print(bool(Playlist('empty', [])))


# Python не проверяет, наследуется ли Playlist от list. Если объект ведет себя как последовательность, он и есть последовательность.
class Playlist:
    def __init__(self, name, songs):
        self.name = name
        self._songs = list(songs)

    def __len__(self):
        return len(self._songs)

    def __iter__(self):
        print('  вызван __iter__')
        yield from self._songs

    def __contains__(self, song):
        print('  вызван __contains__')
        return song.lower() in (s.lower() for s in self._songs)


p = Playlist('road', ['Song A', 'Song B'])
print('song b' in p)
print(list(p))

In [ ]:
# объект, который можно вызвать
class Multiplier:
    def __init__(self, k):
        self.k = k

    def __call__(self, x):
        return x * self.k

double = Multiplier(2)
print(double(21), callable(double), list(map(double, [1, 2, 3])))

### ДЗ 3

Задание 1. Ленивая обработка выгрузки продаж

Напишите набор функций-этапов для обработки CSV-выгрузки продаж. Файл может быть больше оперативной памяти, поэтому все этапы, кроме итоговых агрузок, должны быть генераторами. Модуль csv использовать можно, но и split(',') достаточно.

Формат файла: первая непустая и не-комментарная строка это заголовок date,category,product,qty,price. Строки, начинающиеся с #, и пустые строки пропускаются. Файл упорядочен по дате.

Функции:

read_sales(path, errors=None): генератор словарей вида {'date': '2026-10-01', 'category': 'food', 'product': 'tea', 'qty': 3, 'price': 120.5}. Битые строки (не то число полей, нечисловые qty или price) пропускаются. Если передан список errors, в него добавляется пара (номер строки в файле, текст строки);
daily_revenue(rows): генератор пар (дата, выручка за день), выручка округлена до 2 знаков. Используйте то, что файл упорядочен по дате;
running_revenue(daily): генератор пар (дата, выручка нарастающим итогом);
revenue_by_category(rows): словарь {категория: выручка};
best_days(daily, n=3): список из n самых прибыльных дней, по убыванию выручки. Подумайте, как сделать это без сортировки всех дней (подсказка: модуль heapq).
```python
SAMPLE = '''# выгрузка продаж
date,category,product,qty,price

2026-10-01,books,Dune,2,500
2026-10-01,food,tea,3,120.5
2026-10-02,books,Solaris,1,450
# комментарий посреди файла
2026-10-02,food,coffee,много,300
2026-10-02,toys,ball,4,250
2026-10-03,food,tea,10,120.5
2026-10-03,books,Dune
'''

if __name__ == '__main__':
    import types
    with open('sales_sample.csv', 'w', encoding='utf-8') as f:
        f.write(SAMPLE)

    errors = []
    rows = list(read_sales('sales_sample.csv', errors))
    assert len(rows) == 5
    assert rows[1] == {'date': '2026-10-01', 'category': 'food', 'product': 'tea', 'qty': 3, 'price': 120.5}
    assert [no for no, _ in errors] == [8, 11]

    daily = list(daily_revenue(read_sales('sales_sample.csv')))
    assert daily == [('2026-10-01', 1361.5), ('2026-10-02', 1450.0), ('2026-10-03', 1205.0)]
    assert list(running_revenue(iter(daily))) == [
        ('2026-10-01', 1361.5), ('2026-10-02', 2811.5), ('2026-10-03', 4016.5)]
    assert revenue_by_category(read_sales('sales_sample.csv')) == {
        'books': 1450.0, 'food': 1566.5, 'toys': 1000.0}
    assert best_days(daily_revenue(read_sales('sales_sample.csv')), 2) == [
        ('2026-10-02', 1450.0), ('2026-10-01', 1361.5)]
    assert isinstance(read_sales('sales_sample.csv'), types.GeneratorType)
    assert isinstance(daily_revenue([]), types.GeneratorType)
```
Проверка на большом файле. Сгенерируйте выгрузку за год (около 1 млн строк, 30 МБ) скриптом ниже и измерьте через tracemalloc пик памяти для best_days(daily_revenue(read_sales('sales_big.csv'))). Пик должен быть меньше 1 МБ. Результат замера запишите в комментарий.

```python
import random
from datetime import date, timedelta

def make_big(path='sales_big.csv', days=365, rows_per_day=3000):
    random.seed(0)
    cats = {'books': ['Dune', 'Solaris'], 'food': ['tea', 'coffee'], 'toys': ['ball', 'lego']}
    start = date(2025, 1, 1)
    with open(path, 'w', encoding='utf-8') as f:
        f.write('date,category,product,qty,price\n')
        for d in range(days):
            day = (start + timedelta(days=d)).isoformat()
            for _ in range(rows_per_day):
                c = random.choice(list(cats))
                qty = random.randint(1, 5) if random.random() > 0.001 else 'oops'
                f.write(f'{day},{c},{random.choice(cats[c])},{qty},{random.randint(50, 900)}\n')
```
В комментарии ответьте: почему daily_revenue дал бы неправильный результат на неупорядоченном файле, и как бы вы решили задачу в этом случае (какой ценой по памяти)?

### Задание 2. Свой диапазон дат

### Напишите класс DateRange(start, stop, step=1), аналог range для дат: от start включительно до stop не включительно, step в днях. Это повторение протокола итерации (лекция 6) и протокола контейнера (лекция 7).

Требования:

по объекту можно пройти сколько угодно раз, и вложенные циклы по одному объекту работают правильно;
len(r), r[i] с отрицательными индексами и IndexError за пределами;
d in r работает за O(1), без перебора дат (проверка ниже на диапазоне в 3.6 млн дней должна выполняться мгновенно);
reversed(r) работает без создания списка всех дат;
step <= 0 вызывает ValueError;
__repr__ и __eq__.
```python
if __name__ == '__main__':
    from datetime import date
    r = DateRange(date(2026, 10, 1), date(2026, 10, 11), 3)
    assert list(r) == [date(2026, 10, 1), date(2026, 10, 4), date(2026, 10, 7), date(2026, 10, 10)]
    assert list(r) == list(r)
    assert sum(1 for a in r for b in r) == 16
    assert len(r) == 4
    assert len(DateRange(date(2026, 1, 1), date(2026, 1, 1))) == 0
    assert len(DateRange(date(2026, 1, 5), date(2026, 1, 1))) == 0
    assert r[0] == date(2026, 10, 1) and r[-1] == date(2026, 10, 10)
    try:
        r[4]
        assert False, 'ожидали IndexError'
    except IndexError:
        pass
    assert date(2026, 10, 7) in r
    assert date(2026, 10, 8) not in r and date(2026, 10, 13) not in r
    assert list(reversed(r))[0] == date(2026, 10, 10)
    assert repr(r) == 'DateRange(datetime.date(2026, 10, 1), datetime.date(2026, 10, 11), 3)'
    assert DateRange(date(2026, 1, 1), date(2026, 1, 3)) == DateRange(date(2026, 1, 1), date(2026, 1, 3))
    big = DateRange(date(1, 1, 1), date(9999, 12, 31))
    assert date(2026, 10, 17) in big and len(big) == 3652058
```
В комментарии объясните: чем ваш класс отличается от варианта, где __iter__ возвращает self, и какой баг был бы во втором варианте.

2*. Поддержите срезы: r[1:3] и r[::2] возвращают новый DateRange, а не список. Подсказка: посмотрите, что возвращает range(len(self))[срез].

```python
    s = r[1:3]
    assert isinstance(s, DateRange) and list(s) == [date(2026, 10, 4), date(2026, 10, 7)]
    assert list(r[::2]) == [date(2026, 10, 1), date(2026, 10, 7)]
```

### Задание 3. Ревью кода от LLM

Студент попросил ассистента написать модуль отчета по заказам. Код выглядит аккуратно, у функций есть docstring, но в нем не меньше шести проблем. Часть из них роняет программу, часть молча дает неправильный результат, а одна станет проблемой только на большом файле.

Скопируйте код в hw6/report.py без изменений.

```python
from itertools import groupby


def read_orders(path):
    """Лениво читает заказы из файла: на каждой строке user,amount."""
    with open(path, encoding='utf-8') as f:
        return (line.strip().split(',') for line in f if line.strip())


def totals_by_user(orders):
    """Сумма заказов по каждому пользователю."""
    result = {}
    for user, group in groupby(orders, key=lambda o: o[0]):
        result[user] = sum(float(amount) for _, amount in group)
    return result


def has_large_orders(orders, threshold=1000):
    """Есть ли хотя бы один заказ дороже threshold."""
    large = (o for o in orders if float(o[1]) > threshold)
    return bool(large)


def vip_over_limit(totals, vip_users, vip_limits):
    """VIP-клиенты, превысившие свой лимит."""
    return [u for u, limit in zip(vip_users, vip_limits) if totals.get(u, 0) > limit]


def last_orders(path, n=5):
    """Последние n строк файла заказов."""
    with open(path, encoding='utf-8') as f:
        lines = f.readlines()
    return [line.strip() for line in lines[-n:]]


def build_report(path, vip_users, vip_limits):
    orders = read_orders(path)
    count = sum(1 for _ in orders)
    totals = totals_by_user(orders)
    return {
        'orders': count,
        'totals': totals,
        'has_large': has_large_orders(orders),
        'vip_over_limit': vip_over_limit(totals, vip_users, vip_limits),
        'last': last_orders(path, 3),
    }
```
Тестовые данные:

```python
with open('orders.txt', 'w', encoding='utf-8') as f:
    f.write('anna,100\nbob,50\n\nanna,700\nkate,300\nbob,30\n')
report = build_report('orders.txt', ['anna', 'bob', 'kate'], [500, 10])
```
Сдать нужно три файла.

hw6/review.md: таблица найденных проблем. Для каждой укажите:
функцию;
что не так;
конкретный сценарий: какие входные данные и какой неправильный результат или ошибка;
серьезность: падение, тихо неверный результат или производительность.
hw6/report_fixed.py: исправленная версия с тем же интерфейсом функций. Для last_orders найдите решение, которое не читает весь файл в память. Подсказка: collections.deque с параметром maxlen.
hw6/test_report.py: для каждой проблемы одна проверка, которая падает на report.py и проходит на report_fixed.py. Модуль подключайте через переменную, чтобы прогнать тесты на обеих версиях:
```python
import importlib, sys
module = importlib.import_module(sys.argv[1] if len(sys.argv) > 1 else 'report_fixed')
```
Ожидаемый правильный отчет на тестовых данных: 5 заказов, суммы {'anna': 800.0, 'bob': 80.0, 'kate': 300.0}, крупных заказов нет, VIP с превышением лимита: ['anna', 'bob']. Обратите внимание, что для Кейт лимит не указан. Решите, как правильно себя вести в этом случае, и обоснуйте решение в review.md.